# Module 0.2: Tokenization and Token Economics

<span class="badge blue">Agentic AI</span> <span class="badge amber">~15 min</span> <span class="badge mint">Hands-on</span>

## Learning Objectives

By completing this notebook, you will:
1. Understand how text is converted to tokens
2. Explore different tokenization strategies across models
3. Calculate token costs for real-world applications
4. Optimize prompts for token efficiency
5. Manage context windows effectively

## Prerequisites

- Module 0.1 completed (API setup)
- Understanding of basic string operations
- Familiarity with Python dictionaries and lists

## Estimated Time: 50 minutes

---

## 1. Setup & Imports

We'll use the `tiktoken` library (OpenAI's tokenizer) to explore how text becomes tokens. This gives us precise token counts before making API calls.

In [ ]:
# Setup: Course styling and plot theme
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname("__file__"), "../../../../.."))

from resources.notebook_style import apply_course_theme
from resources.graphics.plot_theme import apply_plot_theme

apply_course_theme()
apply_plot_theme()
print("Course theme applied.")

In [ ]:
# Install required packages
# Uncomment the line below if running for the first time
# !pip install tiktoken anthropic openai

import tiktoken
import json
from typing import List, Dict, Tuple
from collections import Counter

print("✓ Imports successful")

## 2. What Are Tokens?

**Tokens are the fundamental units that LLMs process**. They're not quite words, not quite characters:

- Common words: 1 token (`"hello"` → 1 token)
- Uncommon words: Multiple tokens (`"tokenization"` → 2-3 tokens)
- Punctuation: Usually separate tokens
- Numbers: Each digit can be 1+ tokens
- Code: Often more tokens per character than prose

**Why it matters**:
- APIs charge by tokens (not characters or words)
- Context windows are limited by token count
- Token efficiency = cost savings and better performance

In [ ]:
# Purpose: Initialize tokenizer and demonstrate basic tokenization
# Key Concept: Different models use different tokenizers

def explore_tokenization(text: str, model: str = "gpt-3.5-turbo") -> Dict:
    """
    Tokenize text and return detailed analysis.
    
    Parameters
    ----------
    text : str
        Input text to tokenize
    model : str
        Model name for tokenizer selection
    
    Returns
    -------
    dict
        Contains tokens, count, and analysis
    """
    # Get the appropriate tokenizer for the model
    encoding = tiktoken.encoding_for_model(model)
    
    # Encode text to tokens
    tokens = encoding.encode(text)
    
    # Decode each token to see what it represents
    token_strings = [encoding.decode([token]) for token in tokens]
    
    return {
        "text": text,
        "token_count": len(tokens),
        "tokens": tokens,
        "token_strings": token_strings,
        "char_count": len(text),
        "tokens_per_char": len(tokens) / len(text) if text else 0
    }

# Example 1: Simple English sentence
example1 = explore_tokenization("Hello, world!")
print(f"Text: '{example1['text']}'")
print(f"Tokens: {example1['token_count']}")
print(f"Token breakdown: {example1['token_strings']}")
print()

# Example 2: Technical term
example2 = explore_tokenization("Tokenization")
print(f"Text: '{example2['text']}'")
print(f"Tokens: {example2['token_count']}")
print(f"Token breakdown: {example2['token_strings']}")
print()

# Example 3: Code snippet
example3 = explore_tokenization("def hello(): return 'world'")
print(f"Text: '{example3['text']}'")
print(f"Tokens: {example3['token_count']}")
print(f"Token breakdown: {example3['token_strings']}")

## 3. Token Patterns Across Content Types

Different types of content have different token densities. Understanding this helps you:
- Estimate costs before implementation
- Choose appropriate models for your use case
- Optimize prompt structure

Let's analyze various content types.

In [ ]:
# Purpose: Compare tokenization across different content types
# Key Concept: Token density varies significantly by content type

def analyze_content_types():
    """
    Analyze token patterns across different content types.
    
    Returns
    -------
    dict
        Comparison of token metrics by content type
    """
    samples = {
        "simple_prose": "The quick brown fox jumps over the lazy dog.",
        "technical_prose": "Tokenization algorithms utilize byte-pair encoding for subword segmentation.",
        "python_code": "def fibonacci(n):\n    return n if n <= 1 else fibonacci(n-1) + fibonacci(n-2)",
        "json_data": '{"name": "John", "age": 30, "city": "New York"}',
        "numbers": "1234567890 9876543210 1111111111",
        "mixed": "API key: sk-abc123XYZ, cost: $0.002 per 1K tokens"
    }
    
    results = {}
    
    for content_type, text in samples.items():
        analysis = explore_tokenization(text)
        
        results[content_type] = {
            "text_length": len(text),
            "token_count": analysis['token_count'],
            "tokens_per_char": round(analysis['tokens_per_char'], 3),
            "chars_per_token": round(len(text) / analysis['token_count'], 2)
        }
    
    return results

# Analyze different content types
comparison = analyze_content_types()

print("Token Density by Content Type:")
print("=" * 70)
print(f"{'Content Type':<20} {'Chars':<8} {'Tokens':<8} {'Chars/Token':<12}")
print("=" * 70)

for content_type, metrics in comparison.items():
    print(f"{content_type:<20} {metrics['text_length']:<8} {metrics['token_count']:<8} {metrics['chars_per_token']:<12}")

print("\n💡 Key Insight: Code and technical terms use more tokens per character than simple prose.")

## 4. Context Window Management

Every LLM has a **context window**—the maximum number of tokens it can process (input + output combined).

| Model | Context Window | Input Cost | Output Cost |
|-------|----------------|------------|--------------|
| GPT-3.5-turbo | 16K | $0.50/1M | $1.50/1M |
| GPT-4-turbo | 128K | $10.00/1M | $30.00/1M |
| Claude 3.5 Sonnet | 200K | $3.00/1M | $15.00/1M |

**Context management strategies**:
- Summarize long conversations
- Use retrieval (RAG) instead of full context
- Truncate older messages
- Reserve tokens for output

In [ ]:
# Purpose: Calculate if content fits in context window
# Key Concept: Must budget tokens for both input AND output

def check_context_fit(
    messages: List[Dict[str, str]],
    max_output_tokens: int,
    context_window: int = 16000,
    model: str = "gpt-3.5-turbo"
) -> Dict:
    """
    Check if messages fit within context window with room for output.
    
    Parameters
    ----------
    messages : list of dict
        Conversation messages with 'role' and 'content' keys
    max_output_tokens : int
        Desired maximum output length
    context_window : int
        Model's total context window size
    model : str
        Model name for tokenizer
    
    Returns
    -------
    dict
        Analysis of token usage and fit status
    """
    encoding = tiktoken.encoding_for_model(model)
    
    # Count tokens in all messages
    total_input_tokens = 0
    
    for message in messages:
        # Format: role and content both consume tokens
        role_tokens = len(encoding.encode(message.get('role', '')))
        content_tokens = len(encoding.encode(message.get('content', '')))
        
        # Add 4 tokens per message for formatting (approximate)
        total_input_tokens += role_tokens + content_tokens + 4
    
    # Calculate available space
    available_for_output = context_window - total_input_tokens
    fits = available_for_output >= max_output_tokens
    
    return {
        "input_tokens": total_input_tokens,
        "requested_output_tokens": max_output_tokens,
        "total_required": total_input_tokens + max_output_tokens,
        "context_window": context_window,
        "available_for_output": available_for_output,
        "fits": fits,
        "utilization_pct": round((total_input_tokens + max_output_tokens) / context_window * 100, 1)
    }

# Example: Check if a conversation fits
conversation = [
    {"role": "system", "content": "You are a helpful assistant that explains technical concepts."},
    {"role": "user", "content": "Explain how transformers work in detail."},
    {"role": "assistant", "content": "Transformers are neural network architectures...[long response]"},
    {"role": "user", "content": "Can you give me a code example?"}
]

fit_check = check_context_fit(conversation, max_output_tokens=1000)

print("Context Window Analysis:")
print("=" * 50)
print(f"Input tokens: {fit_check['input_tokens']}")
print(f"Requested output: {fit_check['requested_output_tokens']}")
print(f"Total required: {fit_check['total_required']}")
print(f"Context window: {fit_check['context_window']}")
print(f"Utilization: {fit_check['utilization_pct']}%")
print(f"Fits: {'✓ Yes' if fit_check['fits'] else '✗ No'}")

if not fit_check['fits']:
    print(f"\n⚠️  Need to reduce input by {fit_check['total_required'] - fit_check['context_window']} tokens")

## 5. Token Cost Calculation

Understanding costs is crucial for production systems. Let's build a comprehensive cost calculator that tracks:
- Per-request costs
- Cumulative costs
- Cost comparisons across models

In [ ]:
# Purpose: Calculate and track API costs
# Key Concept: Different models have different pricing tiers

class TokenCostCalculator:
    """Track and calculate LLM API costs."""
    
    # Pricing per 1M tokens (as of 2024)
    PRICING = {
        "gpt-3.5-turbo": {"input": 0.50, "output": 1.50},
        "gpt-4-turbo": {"input": 10.00, "output": 30.00},
        "gpt-4": {"input": 30.00, "output": 60.00},
        "claude-3-5-sonnet": {"input": 3.00, "output": 15.00},
        "claude-3-opus": {"input": 15.00, "output": 75.00},
    }
    
    def __init__(self):
        self.history = []
    
    def calculate_request_cost(
        self,
        model: str,
        input_tokens: int,
        output_tokens: int
    ) -> Dict:
        """
        Calculate cost for a single API request.
        
        Parameters
        ----------
        model : str
            Model identifier
        input_tokens : int
            Number of input tokens
        output_tokens : int
            Number of output tokens
        
        Returns
        -------
        dict
            Cost breakdown
        """
        if model not in self.PRICING:
            raise ValueError(f"Unknown model: {model}. Available: {list(self.PRICING.keys())}")
        
        pricing = self.PRICING[model]
        
        # Calculate costs
        input_cost = (input_tokens / 1_000_000) * pricing["input"]
        output_cost = (output_tokens / 1_000_000) * pricing["output"]
        total_cost = input_cost + output_cost
        
        # Store in history
        record = {
            "model": model,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_tokens": input_tokens + output_tokens,
            "input_cost": input_cost,
            "output_cost": output_cost,
            "total_cost": total_cost
        }
        self.history.append(record)
        
        return record
    
    def get_total_cost(self) -> float:
        """Get cumulative cost across all requests."""
        return sum(record["total_cost"] for record in self.history)
    
    def get_total_tokens(self) -> int:
        """Get cumulative token usage."""
        return sum(record["total_tokens"] for record in self.history)
    
    def compare_models(
        self,
        input_tokens: int,
        output_tokens: int
    ) -> Dict[str, float]:
        """
        Compare costs across all available models.
        
        Parameters
        ----------
        input_tokens : int
            Number of input tokens
        output_tokens : int
            Number of output tokens
        
        Returns
        -------
        dict
            Model names mapped to costs
        """
        costs = {}
        
        for model in self.PRICING.keys():
            pricing = self.PRICING[model]
            cost = (
                (input_tokens / 1_000_000) * pricing["input"] +
                (output_tokens / 1_000_000) * pricing["output"]
            )
            costs[model] = cost
        
        return costs

# Example usage
calculator = TokenCostCalculator()

# Calculate cost for a request
request1 = calculator.calculate_request_cost(
    model="gpt-3.5-turbo",
    input_tokens=500,
    output_tokens=200
)

print("Single Request Cost:")
print(f"Model: {request1['model']}")
print(f"Input: {request1['input_tokens']} tokens (${request1['input_cost']:.6f})")
print(f"Output: {request1['output_tokens']} tokens (${request1['output_cost']:.6f})")
print(f"Total: ${request1['total_cost']:.6f}")
print()

# Compare across models
comparison = calculator.compare_models(input_tokens=10000, output_tokens=2000)

print("Cost Comparison (10K input + 2K output):")
print("=" * 50)
for model, cost in sorted(comparison.items(), key=lambda x: x[1]):
    print(f"{model:<25} ${cost:.4f}")

## Hands-On Exercises

Apply your knowledge of tokenization and cost management.

### Exercise 2.1: Token-Efficient Prompt Rewriting

**Task**: Create a function that shortens prompts by removing unnecessary tokens while preserving meaning.

**Expected Output**: Function reduces token count by at least 10% without losing core information.

**Techniques**:
- Remove redundant phrases ("please", "I would like to")
- Use shorter synonyms
- Eliminate extra whitespace

**Hints**:
<details>
<summary>Hint 1</summary>
Create a list of common verbose phrases and their concise alternatives.
</details>

<details>
<summary>Hint 2</summary>
Use string.replace() for substitutions, then strip extra whitespace.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

def optimize_prompt(prompt: str) -> Tuple[str, int, int]:
    """
    Optimize prompt for token efficiency.
    
    Parameters
    ----------
    prompt : str
        Original prompt
    
    Returns
    -------
    tuple
        (optimized_prompt, original_tokens, optimized_tokens)
    """
    # TODO: Implement prompt optimization
    pass

exercise_2_1_answer = optimize_prompt  # Don't change this line

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_2_1():
    assert exercise_2_1_answer is not None, "Don't forget to implement your solution!"
    
    # Test with verbose prompt
    verbose_prompt = "I would like to kindly request that you please explain to me how tokenization works"
    optimized, original_tokens, new_tokens = exercise_2_1_answer(verbose_prompt)
    
    assert isinstance(optimized, str), "First return value should be a string"
    assert isinstance(original_tokens, int), "Second return value should be an integer"
    assert isinstance(new_tokens, int), "Third return value should be an integer"
    
    # Should reduce tokens
    assert new_tokens < original_tokens, "Optimized version should use fewer tokens"
    
    # Should preserve key terms
    assert "tokenization" in optimized.lower() or "token" in optimized.lower(), \
        "Should preserve key terms like 'tokenization'"
    
    # Should be at least 10% reduction
    reduction = (original_tokens - new_tokens) / original_tokens
    assert reduction >= 0.05, f"Should reduce tokens by at least 5%, got {reduction*100:.1f}%"
    
    print("✓ Exercise 2.1 passed!")
    print(f"  Token reduction: {original_tokens} → {new_tokens} ({reduction*100:.1f}% savings)")

test_exercise_2_1()

### Exercise 2.2: Context Window Budget Manager

**Task**: Create a function that manages conversation history to stay within context limits.

**Expected Output**: Function truncates old messages when total exceeds budget.

**Strategy**: Keep system message and most recent N messages that fit in budget.

**Hints**:
<details>
<summary>Hint 1</summary>
Always preserve the system message (index 0). Start removing from index 1 onwards.
</details>

<details>
<summary>Hint 2</summary>
Work backwards from the most recent messages to keep the freshest context.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

def manage_context_budget(
    messages: List[Dict[str, str]],
    max_tokens: int,
    reserve_for_output: int = 1000
) -> List[Dict[str, str]]:
    """
    Truncate conversation history to fit within token budget.
    
    Parameters
    ----------
    messages : list of dict
        Conversation messages
    max_tokens : int
        Maximum total tokens (context window)
    reserve_for_output : int
        Tokens to reserve for model output
    
    Returns
    -------
    list of dict
        Truncated message list that fits within budget
    """
    # TODO: Implement context budget management
    pass

exercise_2_2_answer = manage_context_budget  # Don't change this line

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_2_2():
    assert exercise_2_2_answer is not None, "Don't forget to implement your solution!"
    
    # Create test messages
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Hello"},
        {"role": "assistant", "content": "Hi there!"},
        {"role": "user", "content": "What's the weather?"},
        {"role": "assistant", "content": "I don't have real-time weather access."},
        {"role": "user", "content": "Tell me a joke"}
    ]
    
    # Test with very small budget (should keep system + most recent messages)
    result = exercise_2_2_answer(messages, max_tokens=100, reserve_for_output=20)
    
    assert isinstance(result, list), "Should return a list"
    assert len(result) > 0, "Should keep at least one message"
    assert len(result) <= len(messages), "Should not add new messages"
    
    # Should preserve system message
    if len(messages) > 0 and messages[0]["role"] == "system":
        assert result[0]["role"] == "system", "Should preserve system message"
    
    # Calculate tokens in result
    encoding = tiktoken.encoding_for_model("gpt-3.5-turbo")
    total_tokens = sum(len(encoding.encode(m["content"])) for m in result)
    
    assert total_tokens <= 100 - 20, f"Result should fit in budget: {total_tokens} tokens (max: 80)"
    
    print("✓ Exercise 2.2 passed!")
    print(f"  Kept {len(result)}/{len(messages)} messages within budget")

test_exercise_2_2()

### Exercise 2.3: Cost Projection Calculator

**Task**: Create a function that projects monthly costs based on usage patterns.

**Expected Output**: Given average requests/day, input/output tokens, calculate monthly cost.

**Hints**:
<details>
<summary>Hint 1</summary>
Monthly cost = (requests_per_day * 30) * cost_per_request
</details>

<details>
<summary>Hint 2</summary>
Use the TokenCostCalculator.PRICING dictionary for model pricing.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

def project_monthly_cost(
    model: str,
    requests_per_day: int,
    avg_input_tokens: int,
    avg_output_tokens: int
) -> Dict:
    """
    Project monthly API costs based on usage patterns.
    
    Parameters
    ----------
    model : str
        Model identifier
    requests_per_day : int
        Average number of requests per day
    avg_input_tokens : int
        Average input tokens per request
    avg_output_tokens : int
        Average output tokens per request
    
    Returns
    -------
    dict
        Projection with daily, monthly costs and total tokens
    """
    # TODO: Implement cost projection
    pass

exercise_2_3_answer = project_monthly_cost  # Don't change this line

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_2_3():
    assert exercise_2_3_answer is not None, "Don't forget to implement your solution!"
    
    # Test projection
    result = exercise_2_3_answer(
        model="gpt-3.5-turbo",
        requests_per_day=1000,
        avg_input_tokens=500,
        avg_output_tokens=200
    )
    
    assert isinstance(result, dict), "Should return a dictionary"
    assert "monthly_cost" in result, "Should include 'monthly_cost' key"
    assert "daily_cost" in result, "Should include 'daily_cost' key"
    
    # Validate calculations
    # 1000 requests * 30 days * (500 input + 200 output) = 21M tokens/month
    # Cost = (15M input * 0.50/1M) + (6M output * 1.50/1M) = $7.50 + $9.00 = $16.50
    expected_monthly = 16.50
    assert abs(result["monthly_cost"] - expected_monthly) < 0.50, \
        f"Expected ~${expected_monthly}, got ${result['monthly_cost']}"
    
    assert result["monthly_cost"] == result["daily_cost"] * 30, \
        "Monthly cost should be 30x daily cost"
    
    print("✓ Exercise 2.3 passed!")
    print(f"  Projected monthly cost: ${result['monthly_cost']:.2f}")

test_exercise_2_3()

## Solutions

Reference implementations for all exercises.

In [ ]:
# SOLUTION 2.1: Token-Efficient Prompt Rewriting

def optimize_prompt_solution(prompt: str) -> Tuple[str, int, int]:
    """
    Optimize prompt for token efficiency.
    """
    encoding = tiktoken.encoding_for_model("gpt-3.5-turbo")
    original_tokens = len(encoding.encode(prompt))
    
    # Common verbose phrases and their concise alternatives
    replacements = {
        "I would like to": "",
        "please": "",
        "kindly": "",
        "Could you": "",
        "Can you": "",
        "I would appreciate if you could": "",
        "explain to me": "explain",
        "tell me about": "explain",
    }
    
    optimized = prompt
    for verbose, concise in replacements.items():
        optimized = optimized.replace(verbose, concise)
    
    # Clean up extra whitespace
    optimized = " ".join(optimized.split())
    optimized = optimized.strip()
    
    optimized_tokens = len(encoding.encode(optimized))
    
    return optimized, original_tokens, optimized_tokens

# SOLUTION 2.2: Context Window Budget Manager

def manage_context_budget_solution(
    messages: List[Dict[str, str]],
    max_tokens: int,
    reserve_for_output: int = 1000
) -> List[Dict[str, str]]:
    """
    Truncate conversation history to fit within token budget.
    """
    encoding = tiktoken.encoding_for_model("gpt-3.5-turbo")
    budget = max_tokens - reserve_for_output
    
    # Always keep system message if present
    result = []
    current_tokens = 0
    
    # Keep system message
    if messages and messages[0]["role"] == "system":
        result.append(messages[0])
        current_tokens += len(encoding.encode(messages[0]["content"])) + 4
        messages = messages[1:]
    
    # Add messages from most recent, working backwards
    for message in reversed(messages):
        message_tokens = len(encoding.encode(message["content"])) + 4
        if current_tokens + message_tokens <= budget:
            result.insert(1 if result and result[0]["role"] == "system" else 0, message)
            current_tokens += message_tokens
        else:
            break
    
    return result

# SOLUTION 2.3: Cost Projection Calculator

def project_monthly_cost_solution(
    model: str,
    requests_per_day: int,
    avg_input_tokens: int,
    avg_output_tokens: int
) -> Dict:
    """
    Project monthly API costs based on usage patterns.
    """
    calculator = TokenCostCalculator()
    
    # Calculate cost per request
    request_cost = calculator.calculate_request_cost(
        model=model,
        input_tokens=avg_input_tokens,
        output_tokens=avg_output_tokens
    )
    
    # Project to daily and monthly
    daily_cost = request_cost["total_cost"] * requests_per_day
    monthly_cost = daily_cost * 30
    
    monthly_requests = requests_per_day * 30
    monthly_tokens = (avg_input_tokens + avg_output_tokens) * monthly_requests
    
    return {
        "model": model,
        "requests_per_day": requests_per_day,
        "daily_cost": daily_cost,
        "monthly_cost": monthly_cost,
        "monthly_requests": monthly_requests,
        "monthly_tokens": monthly_tokens
    }

## Summary

**Key Takeaways**:

1. **Tokenization Varies**: Different content types have different token densities—code is denser than prose
2. **Context Windows**: Always budget tokens for BOTH input and output; don't max out the context
3. **Cost Management**: Token efficiency directly impacts costs—optimize prompts and choose models wisely
4. **Production Planning**: Project costs before deployment to avoid surprises
5. **Model Selection**: Cheaper models (GPT-3.5) can be 20-60x less expensive than premium ones (GPT-4)

**What's Next**:
- Module 0.3: Advanced prompt engineering
- Module 1: Building basic agent loops
- Module 2: Tool use and function calling

**Additional Resources**:
- [OpenAI Tokenizer Tool](https://platform.openai.com/tokenizer) - Interactive tokenization visualization
- [tiktoken GitHub](https://github.com/openai/tiktoken) - Official tokenization library
- [Token Optimization Guide](https://help.openai.com/en/articles/6654000-best-practices-for-prompt-engineering-with-openai-api)

---

Well done! You now understand the economics of LLM APIs and can make informed decisions about token usage.